# SalesInsight Agent

**Natural-language-to-SQL data analysis agent** dengan guardrail keamanan, self-correction, visualisasi otomatis, ringkasan naratif yang ter-*grounding*, dan evaluasi berbasis *execution accuracy*.

> Notebook ini dirancang untuk dijalankan **dari atas ke bawah** (Run All). Bagian yang membutuhkan LLM akan otomatis di-*skip* dengan aman jika `OPENAI_API_KEY` belum diset, sehingga seluruh bagian deterministik (dataset, database, guardrail, security test) tetap bisa dijalankan tanpa biaya.

**Penulis:** Rezky Desyafa · **Versi:** 1.0 · **Level:** Intermediate portfolio project

## 1. Ringkasan Proyek

SalesInsight Agent mengubah pertanyaan bahasa natural menjadi analisis data penjualan. Alurnya:

1. Pengguna bertanya, misal *"Bandingkan revenue kategori Electronics pada Q1 dan Q2 2025."*
2. Agent membaca **schema** dan **aturan bisnis**.
3. LLM menghasilkan **SQL** dalam bentuk *structured output* (Pydantic).
4. SQL divalidasi **sebelum** dieksekusi (SELECT-only, single statement, whitelist tabel, forced LIMIT).
5. Query dijalankan di koneksi **read-only** dengan **timeout**.
6. Jika error, agent **memperbaiki** SQL maksimal dua kali.
7. Hasil ditampilkan sebagai **tabel + grafik**, lalu diringkas dengan **ringkasan naratif** yang diverifikasi agar tidak mengarang angka.
8. Setiap eksekusi dicatat: latency, token, retry, dan estimasi biaya.

### Keputusan desain penting

- **Aturan bisnis ditegakkan di lapisan data (SQL VIEW), bukan di prompt.** LLM cukup query dari view `v_completed_sales` yang sudah mem-*filter* order `Completed`. Ini menutup risiko "SQL valid tapi salah bisnis".
- **Validasi SQL berbasis AST (SQLGlot), bukan regex.** Regex mudah tertipu komentar dan string literal; AST tidak.
- **Deterministik.** `temperature=0` untuk generasi SQL dan `seed` tetap untuk dataset, agar hasil dapat direproduksi dan dievaluasi secara adil.

## 2. Arsitektur

```text
Pertanyaan (natural language)
        |
        v
  Schema + Business Context  <-- get_database_schema()
        |
        v
  LLM SQL Generator (structured output, temperature=0)
        |
        v
  SQL Guardrail (AST / SQLGlot)
   |-- SELECT / CTE-SELECT saja
   |-- Single statement
   |-- Table whitelist (termasuk view)
   |-- Forced LIMIT <= 1000
   +-- Blokir semua operasi tulis/berbahaya
        |
        v
  DuckDB Read-Only + Timeout (watchdog thread)
        |
   +----+----+
   |         |
 Error     Sukses
   |         |
   v         v
Self-      DataFrame --> Visualisasi (Matplotlib)
Correction          +--> Ringkasan naratif (grounded, faithfulness-checked)
(max 2x)            +--> Execution trace (latency, token, retry, biaya)
```

## 3. Instalasi

Jalankan sel di bawah bila dependensi belum terpasang. Di lingkungan yang sudah siap, sel ini bisa dilewati.

> Catatan: pada beberapa CPU lama, NumPy 2.x gagal karena butuh baseline `X86_V2`. Gunakan `numpy<2` jika menemui `RuntimeError` terkait baseline optimizations.

In [ ]:
# !pip install -q "numpy<2" duckdb sqlglot "pydantic>=2" faker matplotlib pandas openai python-dotenv
print("Lewati sel ini jika dependensi sudah terpasang.")

## 4. Imports dan Konfigurasi

Semua import dikumpulkan di satu tempat. Konfigurasi LLM dibaca dari *environment* (tidak ada API key yang di-*hardcode*). Notebook mendukung endpoint **OpenAI-compatible** apa pun lewat `OPENAI_BASE_URL`.

In [ ]:
import os, re, json, time, threading, textwrap, random
from dataclasses import dataclass, field, asdict
from typing import Optional

import numpy as np
import pandas as pd
import duckdb
import sqlglot
from sqlglot import expressions as exp
from pydantic import BaseModel, Field
import matplotlib
matplotlib.use("Agg")  # aman untuk headless; ganti ke inline saat interaktif
import matplotlib.pyplot as plt

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Imports OK. numpy", np.__version__, "| duckdb", duckdb.__version__, "| sqlglot", sqlglot.__version__)

### 4.1 Registry Provider LLM (banyak pilihan gratis)

Notebook ini OpenAI-compatible, jadi provider apa pun yang punya endpoint `/chat/completions` bisa dipakai hanya dengan mengganti `base_url`, `model`, dan API key.

**Provider gratis yang direkomendasikan** (semua mendukung `response_format=json_object` untuk structured output):

| Provider | Model gratis | Env API key |
|---|---|---|
| `groq` ⭐ | llama-3.3-70b-versatile | `GROQ_API_KEY` |
| `cerebras` ⭐ | llama-3.3-70b | `CEREBRAS_API_KEY` |
| `gemini` | gemini-2.0-flash | `GEMINI_API_KEY` |
| `openrouter` | llama-3.3-70b-instruct:free | `OPENROUTER_API_KEY` |
| `mistral` | mistral-small-latest | `MISTRAL_API_KEY` |
| `github` | gpt-4o-mini | `GITHUB_TOKEN` |
| `router` | hermes-claude (router pribadi) | `ROUTER_API_KEY` |
| `custom` | atur sendiri via env | `OPENAI_API_KEY` |

**Cara pakai:** set `SALESINSIGHT_PROVIDER` ke salah satu nama di atas dan isi API key-nya. Contoh Groq (paling cepat & gratis):

```bash
export SALESINSIGHT_PROVIDER=groq
export GROQ_API_KEY=gsk_...
```

Untuk provider yang belum terdaftar, pakai `custom` lalu set `OPENAI_BASE_URL`, `SALESINSIGHT_MODEL`, dan `OPENAI_API_KEY` sendiri.

`temperature=0` memastikan SQL deterministik. Harga token hanya untuk estimasi biaya di trace (provider gratis → biaya 0).

In [ ]:
# --- Registry provider: nama -> (base_url, default_model, env_api_key, harga in/out per 1K USD) ---
PROVIDERS = {
    "groq":       ("https://api.groq.com/openai/v1",            "llama-3.3-70b-versatile",              "GROQ_API_KEY",       0.0, 0.0),
    "cerebras":   ("https://api.cerebras.ai/v1",                "llama-3.3-70b",                        "CEREBRAS_API_KEY",   0.0, 0.0),
    "gemini":     ("https://generativelanguage.googleapis.com/v1beta/openai/", "gemini-2.0-flash",     "GEMINI_API_KEY",     0.0, 0.0),
    "openrouter": ("https://openrouter.ai/api/v1",              "meta-llama/llama-3.3-70b-instruct:free","OPENROUTER_API_KEY",0.0, 0.0),
    "mistral":    ("https://api.mistral.ai/v1",                 "mistral-small-latest",                 "MISTRAL_API_KEY",    0.0, 0.0),
    "github":     ("https://models.inference.ai.azure.com",     "gpt-4o-mini",                          "GITHUB_TOKEN",       0.0, 0.0),
    "router":     ("https://router.unitrade.web.id/v1",         "hermes-claude",                        "ROUTER_API_KEY",     0.0, 0.0),
    "openai":     ("https://api.openai.com/v1",                 "gpt-4o-mini",                          "OPENAI_API_KEY",     0.00015, 0.00060),
    # --- CUSTOM PROVIDER: edit baris ini atau override via env ---
    "custom":     (os.getenv("OPENAI_BASE_URL", ""),            os.getenv("SALESINSIGHT_MODEL", "gpt-4o-mini"), "OPENAI_API_KEY", 0.0, 0.0),
}

PROVIDER = os.getenv("SALESINSIGHT_PROVIDER", "custom").lower()
if PROVIDER not in PROVIDERS:
    raise ValueError(f"Provider '{PROVIDER}' tak dikenal. Pilihan: {list(PROVIDERS)}")

_base, _model, _keyenv, _pin, _pout = PROVIDERS[PROVIDER]
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL") or _base or None
MODEL_NAME      = os.getenv("SALESINSIGHT_MODEL", _model)
OPENAI_API_KEY  = os.getenv(_keyenv) or os.getenv("OPENAI_API_KEY")
PRICE_INPUT_PER_1K  = float(os.getenv("SALESINSIGHT_PRICE_IN",  str(_pin)))
PRICE_OUTPUT_PER_1K = float(os.getenv("SALESINSIGHT_PRICE_OUT", str(_pout)))

LLM_ENABLED = bool(OPENAI_API_KEY)

def _make_client():
    if not LLM_ENABLED:
        return None
    from openai import OpenAI
    kwargs = {"api_key": OPENAI_API_KEY}
    if OPENAI_BASE_URL:
        kwargs["base_url"] = OPENAI_BASE_URL
    return OpenAI(**kwargs)

client = _make_client()
print(f"Provider: {PROVIDER} | model: {MODEL_NAME} | base_url: {OPENAI_BASE_URL or 'default'}")
print("LLM:", "AKTIF" if LLM_ENABLED else f"NONAKTIF — set {_keyenv} untuk mengaktifkan")

## 5. Pembuatan Dataset Sintetis (Reproducible & Bercerita)

Dataset dibuat sintetis agar: tidak memakai data pribadi, dapat direproduksi (`SEED` tetap), dan *ground truth* mudah dibuat.

**Prinsip penting:** data acak murni menghasilkan grafik datar dan demo yang hambar. Maka kita **menanamkan cerita** ke dalam data:

- **Tren pertumbuhan** year-over-year (2025 lebih tinggi dari 2024).
- **Musiman**: puncak akhir tahun (Nov–Des), lembah awal tahun.
- **Dominasi regional**: Jawa menyumbang porsi terbesar.
- **Produk bintang**: sebagian produk sengaja tumbuh tajam agar analisis "pertumbuhan tertinggi" bermakna.
- **Status order** realistis: mayoritas `Completed`, sebagian `Cancelled`/`Refunded` (penting untuk menguji aturan bisnis).

In [ ]:
from faker import Faker
fake = Faker("id_ID")
Faker.seed(SEED)

N_CUSTOMERS = 1000
N_PRODUCTS  = 100
N_ORDERS    = 20000
START_YEAR, END_YEAR = 2024, 2025

REGIONS = {
    "Jawa": 0.45, "Sumatra": 0.20, "Kalimantan": 0.12,
    "Sulawesi": 0.10, "Bali Nusra": 0.08, "Papua Maluku": 0.05,
}
CITIES = {
    "Jawa": ["Jakarta", "Bandung", "Surabaya", "Semarang", "Yogyakarta"],
    "Sumatra": ["Medan", "Palembang", "Padang", "Pekanbaru"],
    "Kalimantan": ["Pontianak", "Balikpapan", "Banjarmasin"],
    "Sulawesi": ["Makassar", "Manado", "Palu"],
    "Bali Nusra": ["Denpasar", "Mataram", "Kupang"],
    "Papua Maluku": ["Jayapura", "Ambon", "Sorong"],
}
SEGMENTS = ["Retail", "Wholesale", "Corporate"]
CATEGORIES = ["Electronics", "Fashion", "Home & Living", "Groceries",
              "Beauty", "Sports", "Toys", "Automotive"]
CHANNELS = ["Website", "Mobile", "Marketplace", "Offline"]
PAYMENTS = ["Transfer", "E-Wallet", "Card", "COD"]
STATUS_WEIGHTS = {"Completed": 0.82, "Cancelled": 0.10, "Refunded": 0.08}

def _weighted(mapping):
    keys = list(mapping); w = np.array(list(mapping.values()), float); w /= w.sum()
    return keys, w

### 5.1 Tabel dimensi: customers, categories, products

`products` diberi *growth factor* per produk. Sebagian kecil produk menjadi "bintang" (tumbuh tajam) agar use case *pertumbuhan produk* menghasilkan ranking yang bermakna.

In [ ]:
# customers
reg_keys, reg_w = _weighted(REGIONS)
cust_rows = []
for cid in range(1, N_CUSTOMERS + 1):
    region = np.random.choice(reg_keys, p=reg_w)
    city = random.choice(CITIES[region])
    cust_rows.append({
        "customer_id": cid,
        "customer_name": fake.name(),
        "customer_segment": random.choices(SEGMENTS, weights=[0.6, 0.25, 0.15])[0],
        "city": city,
        "region": region,
        "registered_at": fake.date_between(start_date="-3y", end_date="today"),
    })
customers = pd.DataFrame(cust_rows)

# categories
categories = pd.DataFrame(
    {"category_id": range(1, len(CATEGORIES) + 1), "category_name": CATEGORIES}
)

# products (+ growth_factor tersembunyi untuk membentuk cerita)
prod_rows = []
for pid in range(1, N_PRODUCTS + 1):
    cat_id = random.randint(1, len(CATEGORIES))
    unit_cost = round(random.uniform(10_000, 2_000_000), -2)
    margin = random.uniform(0.15, 0.55)
    unit_price = round(unit_cost * (1 + margin), -2)
    is_star = random.random() < 0.12   # 12% produk "bintang"
    growth = random.uniform(1.6, 2.8) if is_star else random.uniform(0.8, 1.25)
    prod_rows.append({
        "product_id": pid,
        "product_name": f"{random.choice(CATEGORIES)} {fake.word().title()} {random.randint(100,999)}",
        "category_id": cat_id,
        "unit_cost": unit_cost,
        "unit_price": unit_price,
        "_growth_factor": growth,
    })
products = pd.DataFrame(prod_rows)
print("customers", customers.shape, "| categories", categories.shape, "| products", products.shape)

### 5.2 Fakta: orders & order_items (dengan tren + musiman)

Probabilitas sebuah order jatuh pada bulan tertentu dibentuk oleh **faktor musiman** (puncak akhir tahun) dan **faktor tahun** (2025 > 2024). Kuantitas dan pemilihan produk dipengaruhi *growth factor* produk pada 2025, sehingga "produk bintang" benar-benar melonjak di tahun kedua.

In [ ]:
# Bobot musiman per bulan (1..12): lembah awal tahun, puncak Nov-Des
SEASONAL = np.array([0.7,0.75,0.9,0.95,1.0,1.05,1.1,1.05,1.1,1.2,1.45,1.6])
YEAR_FACTOR = {2024: 1.0, 2025: 1.35}  # pertumbuhan YoY

# Bangun distribusi (year, month)
ym = [(y, m) for y in range(START_YEAR, END_YEAR + 1) for m in range(1, 13)]
ym_w = np.array([SEASONAL[m - 1] * YEAR_FACTOR[y] for (y, m) in ym], float)
ym_w /= ym_w.sum()

st_keys, st_w = _weighted(STATUS_WEIGHTS)
prod_ids = products["product_id"].to_numpy()
prod_lookup = products.set_index("product_id")

order_rows, item_rows = [], []
order_item_id = 0
from datetime import date
import calendar

for oid in range(1, N_ORDERS + 1):
    idx = np.random.choice(len(ym), p=ym_w)
    y, m = ym[idx]
    day = random.randint(1, calendar.monthrange(y, m)[1])
    order_date = date(y, m, day)
    status = np.random.choice(st_keys, p=st_w)
    order_rows.append({
        "order_id": oid,
        "customer_id": random.randint(1, N_CUSTOMERS),
        "order_date": order_date,
        "order_status": status,
        "sales_channel": random.choices(CHANNELS, weights=[0.3,0.35,0.25,0.1])[0],
        "payment_method": random.choice(PAYMENTS),
    })
    # 1..5 item per order
    for _ in range(random.randint(1, 5)):
        order_item_id += 1
        pid = int(np.random.choice(prod_ids))
        row = prod_lookup.loc[pid]
        gf = row["_growth_factor"] if y == 2025 else 1.0
        base_qty = np.random.poisson(2) + 1
        qty = max(1, int(round(base_qty * (gf if gf > 1 else 1))))
        unit_price = float(row["unit_price"])
        discount = round(unit_price * qty * random.choice([0, 0, 0, 0.05, 0.1, 0.15]), -2)
        revenue = round(unit_price * qty - discount, -2)
        cost = round(float(row["unit_cost"]) * qty, -2)
        item_rows.append({
            "order_item_id": order_item_id,
            "order_id": oid,
            "product_id": pid,
            "quantity": qty,
            "unit_price": unit_price,
            "discount": discount,
            "revenue": revenue,
            "cost": cost,
            "profit": round(revenue - cost, -2),
        })

orders = pd.DataFrame(order_rows)
order_items = pd.DataFrame(item_rows)
print("orders", orders.shape, "| order_items", order_items.shape)
print("status dist:\n", orders.order_status.value_counts(normalize=True).round(3).to_string())

## 6. Exploratory Data Analysis singkat

Memastikan cerita benar-benar tertanam: revenue 2025 harus di atas 2024, dan ada pola musiman.

In [ ]:
_items = order_items.merge(orders[["order_id","order_date","order_status"]], on="order_id")
_items = _items[_items.order_status == "Completed"].copy()
_items["year"] = pd.to_datetime(_items.order_date).dt.year
_items["month"] = pd.to_datetime(_items.order_date).dt.month
by_year = _items.groupby("year").revenue.sum()
print("Revenue per tahun (Completed):")
print(by_year.map(lambda v: f"Rp{v:,.0f}").to_string())
assert by_year.get(2025, 0) > by_year.get(2024, 0), "Cerita gagal: 2025 harus > 2024"
print("\nOK: tren pertumbuhan YoY tertanam.")

## 7. DuckDB Setup

Semua tabel ditulis ke database file DuckDB. Kolom internal `_growth_factor` **tidak** ikut ditulis (hanya alat pembentuk cerita).

In [ ]:
DB_PATH = "sales.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

_con = duckdb.connect(DB_PATH)  # koneksi write, hanya untuk setup
_con.execute("CREATE TABLE customers  AS SELECT * FROM customers")
_con.execute("CREATE TABLE categories AS SELECT * FROM categories")
_con.execute("CREATE TABLE products   AS SELECT * EXCLUDE (_growth_factor) FROM products")
_con.execute("CREATE TABLE orders      AS SELECT * FROM orders")
_con.execute("CREATE TABLE order_items AS SELECT * FROM order_items")
print("Tabel dibuat:", [r[0] for r in _con.execute("SHOW TABLES").fetchall()])

## 8. Dokumentasi Schema & 9. Definisi Metrik Bisnis (Semantic Views)

Inilah inti keandalan: **aturan bisnis ditegakkan sebagai VIEW**, bukan diserahkan ke LLM.

- `v_completed_sales` — hanya order `Completed`, sudah join ke `orders`. Semua perhitungan revenue/profit sebaiknya dari sini.
- Definisi kuartal, AOV, dan pertumbuhan didokumentasikan agar LLM konsisten.

Dengan cara ini, meski LLM lupa memfilter status, angka dari view tetap benar.

In [ ]:
_con.execute('''
CREATE VIEW v_completed_sales AS
SELECT
    oi.order_item_id, oi.order_id, oi.product_id,
    oi.quantity, oi.unit_price, oi.discount,
    oi.revenue, oi.cost, oi.profit,
    o.customer_id, o.order_date, o.sales_channel, o.payment_method,
    EXTRACT(year  FROM o.order_date) AS year,
    EXTRACT(month FROM o.order_date) AS month,
    EXTRACT(quarter FROM o.order_date) AS quarter
FROM order_items oi
JOIN orders o ON oi.order_id = o.order_id
WHERE o.order_status = 'Completed';
''')
print("View v_completed_sales dibuat.")
_con.close()   # tutup koneksi write; selanjutnya read-only
print("Koneksi write ditutup.")

In [ ]:
SCHEMA_DOC = '''
TABEL & VIEW (semua kolom hanya-baca):

view v_completed_sales  -- GUNAKAN INI untuk semua metrik revenue/profit (hanya order Completed)
  order_item_id, order_id, product_id, quantity, unit_price, discount,
  revenue, cost, profit, customer_id, order_date, sales_channel, payment_method,
  year, month, quarter

table customers(customer_id, customer_name, customer_segment, city, region, registered_at)
table categories(category_id, category_name)
table products(product_id, product_name, category_id, unit_cost, unit_price)
table orders(order_id, customer_id, order_date, order_status, sales_channel, payment_method)
table order_items(order_item_id, order_id, product_id, quantity, unit_price, discount, revenue, cost, profit)

RELASI:
  products.category_id     -> categories.category_id
  orders.customer_id       -> customers.customer_id
  order_items.order_id     -> orders.order_id
  order_items.product_id   -> products.product_id
'''

BUSINESS_RULES = '''
ATURAN BISNIS:
1. Revenue & profit HANYA dari order berstatus Completed. Gunakan view v_completed_sales.
2. Order Cancelled dan Refunded TIDAK dihitung sebagai revenue.
3. Revenue = kolom revenue; Profit = kolom profit (sudah bersih diskon & cost).
4. Average Order Value (AOV) = SUM(revenue) / COUNT(DISTINCT order_id).
5. Kuartal: Q1=Jan-Mar, Q2=Apr-Jun, Q3=Jul-Sep, Q4=Okt-Des (pakai kolom quarter).
6. Pertumbuhan (%) = (akhir - awal) / awal * 100. Jika awal = 0 -> tidak dapat dihitung.
7. Mata uang: Rupiah (IDR).
8. Untuk analisis produk/kategori/wilayah, join view v_completed_sales ke tabel dimensi.
'''
print("Schema & business rules terdokumentasi.")

## 10. Koneksi Database Read-Only

Setelah setup selesai, agent **hanya** membuka DuckDB dalam mode read-only. Ini lapisan pertahanan terluar: bahkan jika guardrail tertembus, engine menolak operasi tulis.

In [ ]:
def get_readonly_connection():
    return duckdb.connect(DB_PATH, read_only=True)

# Uji: koneksi read-only menolak operasi tulis
_ro = get_readonly_connection()
_denied = False
try:
    _ro.execute("CREATE TABLE hack(x INT)")
except Exception as e:
    _denied = True
    _msg = str(e).splitlines()[0]
_ro.close()
assert _denied, "read-only gagal menolak tulis!"
print("OK read-only menolak tulis:", _msg[:80])

## 11. SQL Validation Guardrail (berbasis AST)

Validasi memakai **AST SQLGlot** (dialek `duckdb`), bukan regex. Aturan:

1. **Single statement** — query multi-statement ditolak.
2. **SELECT-only** — hanya `SELECT` atau CTE (`WITH ... SELECT`). Semua DML/DDL diblokir.
3. **Table whitelist** — hanya tabel/view yang diizinkan.
4. **Blokir operasi berbahaya** — INSERT/UPDATE/DELETE/DROP/ALTER/CREATE/ATTACH/COPY/PRAGMA/dst.
5. **Forced LIMIT** — jika tidak ada LIMIT atau LIMIT > 1000, dipaksa menjadi 1000 (dilakukan di AST, aman terhadap komentar/CTE).

Hasil validasi berupa objek terstruktur: `ok`, `reason`, dan `safe_sql` (SQL final yang sudah di-inject LIMIT).

In [ ]:
ALLOWED_TABLES = {
    "customers", "categories", "products", "orders", "order_items",
    "v_completed_sales",
}
MAX_LIMIT = 1000
QUERY_TIMEOUT_SEC = 5

# Node ekspresi yang menandakan operasi menulis/berbahaya
FORBIDDEN_NODES = (
    exp.Insert, exp.Update, exp.Delete, exp.Drop, exp.Create, exp.Alter,
    exp.Command,   # PRAGMA, CALL, COPY, dsb sering diparse sebagai Command
    exp.Merge,
)

@dataclass
class ValidationResult:
    ok: bool
    reason: str = ""
    safe_sql: Optional[str] = None

def _extract_tables(tree) -> set:
    names = set()
    for t in tree.find_all(exp.Table):
        names.add(t.name.lower())
    return names

def _cte_names(tree) -> set:
    names = set()
    for cte in tree.find_all(exp.CTE):
        if cte.alias:
            names.add(cte.alias.lower())
    return names

def validate_sql(sql: str) -> ValidationResult:
    if not sql or not sql.strip():
        return ValidationResult(False, "SQL kosong.")
    # 1) parse semua statement dgn dialek duckdb
    try:
        statements = sqlglot.parse(sql, dialect="duckdb")
    except Exception as e:
        return ValidationResult(False, f"Parse error: {str(e).splitlines()[0]}")
    statements = [s for s in statements if s is not None]
    if len(statements) != 1:
        return ValidationResult(False, f"Harus tepat satu statement (ditemukan {len(statements)}).")
    tree = statements[0]

    # 2) blokir node berbahaya di mana pun dalam pohon
    for node_type in FORBIDDEN_NODES:
        if tree.find(node_type) is not None:
            return ValidationResult(False, f"Operasi dilarang terdeteksi: {node_type.__name__}.")

    # 3) root harus SELECT (langsung atau via WITH)
    root = tree
    if not isinstance(root, (exp.Select,)):
        # WITH ... SELECT diparse sebagai Select dengan args['with']; jika bukan Select -> tolak
        if not (isinstance(root, exp.Subquery) and isinstance(root.this, exp.Select)):
            return ValidationResult(False, f"Hanya SELECT/CTE-SELECT yang diizinkan (root={type(root).__name__}).")

    # 4) whitelist tabel (abaikan nama CTE)
    ctes = _cte_names(tree)
    used = _extract_tables(tree) - ctes
    illegal = used - ALLOWED_TABLES
    if illegal:
        return ValidationResult(False, f"Tabel tidak diizinkan: {sorted(illegal)}.")

    # 5) forced LIMIT di AST
    limit = root.args.get("limit")
    if limit is not None:
        try:
            cur = int(limit.expression.this)
            if cur > MAX_LIMIT:
                root.set("limit", exp.Limit(expression=exp.Literal.number(MAX_LIMIT)))
        except Exception:
            root.set("limit", exp.Limit(expression=exp.Literal.number(MAX_LIMIT)))
    else:
        root.set("limit", exp.Limit(expression=exp.Literal.number(MAX_LIMIT)))

    safe_sql = root.sql(dialect="duckdb")
    return ValidationResult(True, "OK", safe_sql)

print("validate_sql siap.")

### 11.1 Uji cepat guardrail (deterministik)

Beberapa kasus positif/negatif untuk memastikan validator bekerja sebelum menyentuh LLM.

In [ ]:
_cases = [
    ("SELECT SUM(revenue) FROM v_completed_sales", True),
    ("WITH t AS (SELECT year, SUM(revenue) r FROM v_completed_sales GROUP BY year) SELECT * FROM t", True),
    ("SELECT * FROM v_completed_sales LIMIT 999999", True),   # akan dipaksa 1000
    ("DROP TABLE orders", False),
    ("DELETE FROM customers", False),
    ("UPDATE products SET unit_price = 0", False),
    ("SELECT * FROM orders; DROP TABLE orders", False),
    ("SELECT * FROM secret_table", False),
    ("COPY orders TO '/tmp/x.csv'", False),
    ("PRAGMA database_list", False),
]
_pass = 0
for sql, expect in _cases:
    r = validate_sql(sql)
    ok = (r.ok == expect)
    _pass += ok
    tag = "OK " if ok else "XX "
    print(f"{tag} expect={expect!s:5} got={r.ok!s:5}  {sql[:48]}")
print(f"\nGuardrail unit: {_pass}/{len(_cases)} lulus")
assert _pass == len(_cases), "Ada kasus guardrail gagal!"
# cek forced limit benar-benar 1000
_r = validate_sql("SELECT * FROM v_completed_sales LIMIT 999999")
assert "1000" in _r.safe_sql and "999999" not in _r.safe_sql, "Forced LIMIT gagal"
print("Forced LIMIT OK ->", _r.safe_sql)

## 12. Database Query Tool (dengan timeout)

DuckDB tidak memiliki `statement_timeout` bawaan seperti PostgreSQL. Kita menerapkan **watchdog thread** yang memanggil `connection.interrupt()` bila query melewati batas waktu. `execute_sql` mengembalikan metadata lengkap untuk trace.

In [ ]:
@dataclass
class QueryResult:
    status: str                     # 'success' | 'error' | 'timeout'
    dataframe: Optional[pd.DataFrame] = None
    error_type: Optional[str] = None
    error_message: Optional[str] = None
    row_count: int = 0
    execution_time: float = 0.0
    executed_sql: Optional[str] = None

def execute_sql(sql: str, timeout_sec: int = QUERY_TIMEOUT_SEC) -> QueryResult:
    """Validasi lalu eksekusi di koneksi read-only dengan timeout watchdog."""
    v = validate_sql(sql)
    if not v.ok:
        return QueryResult(status="error", error_type="UnsafeSQL", error_message=v.reason, executed_sql=sql)

    safe_sql = v.safe_sql
    con = get_readonly_connection()
    timed_out = {"flag": False}
    def _watchdog():
        timed_out["flag"] = True
        try:
            con.interrupt()
        except Exception:
            pass
    timer = threading.Timer(timeout_sec, _watchdog)
    t0 = time.time()
    try:
        timer.start()
        df = con.execute(safe_sql).fetchdf()
        elapsed = time.time() - t0
        return QueryResult("success", dataframe=df, row_count=len(df),
                           execution_time=elapsed, executed_sql=safe_sql)
    except Exception as e:
        elapsed = time.time() - t0
        if timed_out["flag"]:
            return QueryResult("timeout", error_type="Timeout",
                               error_message=f"Query melebihi {timeout_sec}s",
                               execution_time=elapsed, executed_sql=safe_sql)
        return QueryResult("error", error_type=type(e).__name__,
                           error_message=str(e).splitlines()[0],
                           execution_time=elapsed, executed_sql=safe_sql)
    finally:
        timer.cancel()
        con.close()

# Uji deterministik: query benar mengembalikan data
_r = execute_sql("SELECT year, SUM(revenue) AS revenue FROM v_completed_sales GROUP BY year ORDER BY year")
assert _r.status == "success" and _r.row_count >= 1, _r
print("execute_sql OK:", _r.status, "rows=", _r.row_count)
print(_r.dataframe.assign(revenue=lambda d: d.revenue.map(lambda v: f"Rp{v:,.0f}")).to_string(index=False))
# Uji: unsafe ditolak sebelum eksekusi
_r2 = execute_sql("DROP TABLE orders")
assert _r2.status == "error" and _r2.error_type == "UnsafeSQL"
print("execute_sql menolak unsafe:", _r2.error_message)

## 12. Structured Output (Pydantic)

LLM diminta mengembalikan objek terstruktur, bukan teks bebas. Ini membuat parsing andal dan memungkinkan validasi. `chart` mendeskripsikan visualisasi yang diinginkan; `assumptions` memaksa model mengeksplisitkan asumsinya.

In [ ]:
class ChartConfig(BaseModel):
    required: bool = False
    chart_type: Optional[str] = None      # line|bar|hbar|grouped_bar|scatter
    x: Optional[str] = None
    y: list[str] = Field(default_factory=list)
    title: Optional[str] = None

class SQLGenerationResult(BaseModel):
    question_interpretation: str
    sql: str
    expected_columns: list[str] = Field(default_factory=list)
    chart: ChartConfig = Field(default_factory=ChartConfig)
    assumptions: list[str] = Field(default_factory=list)

print("Skema Pydantic siap.")

## 13. LLM Client Abstraction

Satu fungsi `llm_json()` memanggil endpoint OpenAI-compatible dengan `temperature=0` dan meminta output JSON. Mengembalikan `(dict, usage)` agar token bisa dicatat. Jika LLM nonaktif, fungsi memberi error yang jelas.

In [ ]:
@dataclass
class Usage:
    input_tokens: int = 0
    output_tokens: int = 0
    def cost(self) -> float:
        return (self.input_tokens/1000*PRICE_INPUT_PER_1K
                + self.output_tokens/1000*PRICE_OUTPUT_PER_1K)

def llm_json(system: str, user: str, temperature: float = 0.0):
    if not LLM_ENABLED:
        raise RuntimeError("LLM nonaktif: set OPENAI_API_KEY untuk memakai fitur ini.")
    resp = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=temperature,
        response_format={"type": "json_object"},
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": user}],
    )
    content = resp.choices[0].message.content
    u = resp.usage
    usage = Usage(getattr(u, "prompt_tokens", 0) or 0, getattr(u, "completion_tokens", 0) or 0)
    return json.loads(content), usage

def llm_text(system: str, user: str, temperature: float = 0.0):
    if not LLM_ENABLED:
        raise RuntimeError("LLM nonaktif: set OPENAI_API_KEY.")
    resp = client.chat.completions.create(
        model=MODEL_NAME, temperature=temperature,
        messages=[{"role":"system","content":system},{"role":"user","content":user}],
    )
    u = resp.usage
    return resp.choices[0].message.content, Usage(getattr(u,"prompt_tokens",0) or 0, getattr(u,"completion_tokens",0) or 0)

print("Klien LLM siap (aktif=%s)." % LLM_ENABLED)

## 14. SQL Generation Prompt

Prompt menyuntikkan schema + aturan bisdan + kontrak keamanan, lalu meminta `SQLGenerationResult` sebagai JSON. Instruksi menekankan pemakaian view `v_completed_sales` untuk metrik.

In [ ]:
SQL_SYSTEM = textwrap.dedent(f'''
Anda adalah analis data SQL untuk DuckDB. Ubah pertanyaan pengguna menjadi SATU query SELECT yang aman.

{SCHEMA_DOC}
{BUSINESS_RULES}

KONTRAK KEAMANAN (WAJIB):
- Hanya SELECT atau WITH ... SELECT. Tidak boleh INSERT/UPDATE/DELETE/DROP/ALTER/CREATE/COPY/PRAGMA.
- Tepat satu statement, tanpa tanda ';' berganda.
- Hanya tabel/view yang diizinkan: customers, categories, products, orders, order_items, v_completed_sales.
- Untuk revenue/profit, WAJIB pakai view v_completed_sales.
- Selalu batasi hasil dengan LIMIT <= {MAX_LIMIT}.

Balas HANYA JSON valid dengan skema:
{{"question_interpretation": str, "sql": str, "expected_columns": [str],
  "chart": {{"required": bool, "chart_type": "line|bar|hbar|grouped_bar|scatter|null",
             "x": str|null, "y": [str], "title": str|null}},
  "assumptions": [str]}}
''').strip()

def generate_sql(question: str, retry_context: str = ""):
    user = question if not retry_context else f"{question}\n\n{retry_context}"
    data, usage = llm_json(SQL_SYSTEM, user)
    return SQLGenerationResult(**data), usage

print("Prompt generator SQL siap.")

## 15. Self-Correction (maksimal 2x)

Bila query gagal (error sintaksis/eksekusi/timeout, atau ditolak guardrail), LLM menerima konteks: SQL sebelumnya, pesan error, nomor percobaan, dan kontrak keamanan — lalu memperbaiki **tanpa mengubah maksud** pertanyaan. Hasil koreksi divalidasi ulang. Query berbahaya **tidak** memicu self-correction berulang tanpa batas; total percobaan = 1 awal + maksimal 2 retry.

In [ ]:
MAX_RETRY = 2

def build_retry_context(prev_sql: str, error_msg: str, attempt: int) -> str:
    return textwrap.dedent(f'''
    Percobaan sebelumnya (#{attempt}) GAGAL.
    SQL sebelumnya:
    {prev_sql}
    Pesan error:
    {error_msg}
    Perbaiki SQL agar valid dan aman TANPA mengubah maksud pertanyaan.
    Patuhi kontrak keamanan dan gunakan v_completed_sales untuk metrik.
    ''').strip()
print("Self-correction util siap.")

## 16. Visualization Tool

Membuat grafik Matplotlib sesuai `ChartConfig`. Prinsip: judul jelas, sumbu berlabel, data waktu terurut, dan grafik tidak dibuat bila tidak relevan. Angka besar diformat ringkas.

In [ ]:
def _fmt_rupiah(v, _pos=None):
    for unit, div in [("T",1e12),("M",1e9),("jt",1e6),("rb",1e3)]:
        if abs(v) >= div:
            return f"{v/div:.1f}{unit}"
    return f"{v:.0f}"

def create_visualization(df: pd.DataFrame, cfg: ChartConfig):
    if not cfg.required or df is None or df.empty or not cfg.chart_type:
        return None
    ycols = [c for c in cfg.y if c in df.columns]
    if not ycols or (cfg.x and cfg.x not in df.columns):
        return None
    fig, ax = plt.subplots(figsize=(8, 4.5))
    x = df[cfg.x] if cfg.x else df.index
    ct = cfg.chart_type
    try:
        if ct == "line":
            d = df.sort_values(cfg.x) if cfg.x else df
            for c in ycols: ax.plot(d[cfg.x] if cfg.x else d.index, d[c], marker="o", label=c)
        elif ct == "bar":
            ax.bar(x.astype(str), df[ycols[0]])
        elif ct == "hbar":
            ax.barh(x.astype(str), df[ycols[0]]); ax.invert_yaxis()
        elif ct == "grouped_bar":
            import numpy as _np
            idx = _np.arange(len(df)); w = 0.8/max(len(ycols),1)
            for i,c in enumerate(ycols): ax.bar(idx+i*w, df[c], w, label=c)
            ax.set_xticks(idx+w*(len(ycols)-1)/2); ax.set_xticklabels(x.astype(str), rotation=45, ha="right")
        elif ct == "scatter":
            ax.scatter(df[cfg.x], df[ycols[0]])
        else:
            plt.close(fig); return None
    except Exception as e:
        plt.close(fig); print("Viz error:", e); return None
    ax.set_title(cfg.title or "")
    if cfg.x: ax.set_xlabel(cfg.x)
    ax.set_ylabel(", ".join(ycols))
    if any(k in " ".join(ycols).lower() for k in ["revenue","profit","cost","price","value","aov"]):
        ax.yaxis.set_major_formatter(plt.FuncFormatter(_fmt_rupiah))
    if len(ycols) > 1 or ct in ("line","grouped_bar"): ax.legend()
    if ct not in ("hbar",): plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    return fig
print("create_visualization siap.")

## 17. Narrative Summarization + Faithfulness Check

Ringkasan naratif **hanya** boleh berdasarkan hasil query. Untuk mencegah halusinasi angka, kita menjalankan **faithfulness check**: setiap angka dalam ringkasan diverifikasi keberadaannya pada DataFrame hasil (dengan toleransi pembulatan). Angka yang tidak dapat diverifikasi ditandai.

In [ ]:
def _numbers_in_text(text: str):
    raw = re.findall(r"-?\d[\d.,]*", text or "")
    out = []
    for r in raw:
        s = r.replace(".", "").replace(",", ".") if r.count(",")==1 and r.count(".")>=1 else r.replace(",", "")
        try: out.append(float(s))
        except Exception: pass
    return out

def _df_number_pool(df: pd.DataFrame):
    pool = set()
    for col in df.select_dtypes(include=[np.number]).columns:
        for v in df[col].dropna().tolist():
            pool.add(round(float(v), 2))
    return pool

def faithfulness_check(summary: str, df: pd.DataFrame, tol_ratio: float = 0.02):
    if df is None or df.empty:
        return {"checked": 0, "unverified": [], "score": 1.0}
    pool = _df_number_pool(df)
    pool_list = sorted(pool)
    nums = [n for n in _numbers_in_text(summary) if abs(n) >= 100]  # abaikan angka kecil (tahun, indeks)
    unverified = []
    for n in nums:
        ok = any(abs(n-p) <= max(abs(p)*tol_ratio, 1.0) for p in pool_list)
        # toleransi persentase yang dihitung dari dua nilai pool juga diterima
        if not ok:
            unverified.append(n)
    checked = len(nums)
    score = 1.0 if checked == 0 else 1 - len(unverified)/checked
    return {"checked": checked, "unverified": unverified, "score": round(score,3)}

NARR_SYSTEM = ("Anda analis data. Buat ringkasan singkat (maks 6 kalimat) HANYA berdasarkan tabel hasil. "
               "Sebutkan angka kunci apa adanya. Jangan mengarang data di luar tabel. Bahasa Indonesia.")

def summarize(question: str, df: pd.DataFrame, assumptions: list[str]):
    head = df.head(50).to_markdown(index=False)
    user = f"Pertanyaan: {question}\n\nTabel hasil (maks 50 baris):\n{head}\n\nAsumsi: {assumptions}"
    text, usage = llm_text(NARR_SYSTEM, user)
    fc = faithfulness_check(text, df)
    return text, usage, fc

print("Narrative + faithfulness siap.")

## 18. Agent Orchestration Loop

Menyatukan semuanya. `run_sales_agent()` menjalankan: generate SQL -> validasi+eksekusi -> self-correct (maks 2x) -> visualisasi -> ringkasan -> catat trace. Mengembalikan objek hasil yang rapi dan menyimpan trace ke `agent_traces`.

In [ ]:
agent_traces = []

@dataclass
class AgentResult:
    question: str
    status: str
    interpretation: str = ""
    final_sql: Optional[str] = None
    dataframe: Optional[pd.DataFrame] = None
    figure: object = None
    summary: str = ""
    assumptions: list = field(default_factory=list)
    faithfulness: dict = field(default_factory=dict)
    retries: int = 0
    error: Optional[str] = None
    latency_total: float = 0.0
    latency_query: float = 0.0
    input_tokens: int = 0
    output_tokens: int = 0
    est_cost: float = 0.0

def run_sales_agent(question: str, show_trace: bool = True) -> AgentResult:
    t0 = time.time()
    total_usage = Usage()
    res = AgentResult(question=question, status="error")
    retry_ctx = ""; last_err = ""; gen = None
    if not LLM_ENABLED:
        res.error = "LLM nonaktif (set OPENAI_API_KEY)."; res.latency_total = time.time()-t0
        return res
    for attempt in range(MAX_RETRY + 1):
        try:
            gen, u = generate_sql(question, retry_ctx)
        except Exception as e:
            last_err = f"Generation error: {e}"; break
        total_usage.input_tokens += u.input_tokens; total_usage.output_tokens += u.output_tokens
        res.interpretation = gen.question_interpretation; res.assumptions = gen.assumptions
        q = execute_sql(gen.sql)
        res.latency_query += q.execution_time
        if q.status == "success":
            res.status = "success"; res.final_sql = q.executed_sql
            res.dataframe = q.dataframe; res.retries = attempt
            try:
                res.figure = create_visualization(q.dataframe, gen.chart)
            except Exception as e:
                print("Viz gagal:", e)
            try:
                summary, su, fc = summarize(question, q.dataframe, gen.assumptions)
                total_usage.input_tokens += su.input_tokens; total_usage.output_tokens += su.output_tokens
                res.summary = summary; res.faithfulness = fc
            except Exception as e:
                res.summary = "(ringkasan gagal dibuat)"; print("Summary gagal:", e)
            break
        else:
            last_err = f"{q.error_type}: {q.error_message}"
            retry_ctx = build_retry_context(gen.sql, last_err, attempt + 1)
            res.retries = attempt + 1
    if res.status != "success":
        res.error = last_err or "gagal"
    res.latency_total = time.time() - t0
    res.input_tokens = total_usage.input_tokens; res.output_tokens = total_usage.output_tokens
    res.est_cost = total_usage.cost()
    agent_traces.append({
        "question": question, "status": res.status, "retries": res.retries,
        "final_sql": res.final_sql, "row_count": (0 if res.dataframe is None else len(res.dataframe)),
        "latency_query": round(res.latency_query,3), "latency_total": round(res.latency_total,3),
        "input_tokens": res.input_tokens, "output_tokens": res.output_tokens,
        "est_cost": round(res.est_cost,6),
        "faithfulness": res.faithfulness.get("score") if res.faithfulness else None,
        "error": res.error,
    })
    if show_trace:
        _render_result(res)
    return res

def _render_result(res: AgentResult):
    print("="*70); print("PERTANYAAN:", res.question); print("STATUS:", res.status)
    if res.status != "success":
        print("ERROR:", res.error); return
    print("\nInterpretasi:", res.interpretation)
    if res.dataframe is not None:
        print("\nTabel hasil (maks 20 baris):")
        print(res.dataframe.head(20).to_string(index=False))
    print("\nRingkasan:\n", res.summary)
    if res.faithfulness:
        fc = res.faithfulness
        print(f"\nFaithfulness: score={fc['score']} checked={fc['checked']} unverified={fc['unverified']}")
    print("\nSQL final:\n", res.final_sql)
    if res.assumptions: print("\nAsumsi:", res.assumptions)
    print(f"\nMetadata: retries={res.retries} | query={res.latency_query:.2f}s | "
          f"total={res.latency_total:.2f}s | in_tok={res.input_tokens} out_tok={res.output_tokens} | "
          f"biaya=${res.est_cost:.6f}")
    if res.figure is not None:
        try: res.figure.show()
        except Exception: pass

print("Agent loop siap. Panggil run_sales_agent('pertanyaan Anda').")

## 19. Demonstrasi (10 Pertanyaan)

Sepuluh pertanyaan yang mencakup agregasi, time-series, perbandingan, ranking, dan multi-tabel. Jika LLM aktif, tiap pertanyaan dijalankan penuh; jika tidak, daftar tetap ditampilkan sebagai referensi.

In [ ]:
DEMO_QUESTIONS = [
    "Berapa total revenue pada tahun 2025?",
    "Tampilkan tren revenue bulanan sepanjang tahun 2025.",
    "Bandingkan revenue tiap kategori pada Q1 dan Q2 2025.",
    "Kategori mana yang memiliki profit tertinggi sepanjang 2025?",
    "Wilayah mana dengan revenue tertinggi?",
    "Tampilkan lima produk dengan revenue terbesar.",
    "Bandingkan average order value tiap sales channel.",
    "Siapa sepuluh pelanggan dengan total pembelian terbesar?",
    "Bulan mana pada 2025 yang mengalami penurunan revenue terbesar dibanding bulan sebelumnya?",
    "Bandingkan performa channel Website dan Marketplace berdasarkan revenue.",
]

if LLM_ENABLED:
    for i, q in enumerate(DEMO_QUESTIONS, 1):
        print(f"\n########## DEMO {i}/{len(DEMO_QUESTIONS)} ##########")
        run_sales_agent(q, show_trace=True)
else:
    print("LLM nonaktif — daftar pertanyaan demo:")
    for i, q in enumerate(DEMO_QUESTIONS, 1):
        print(f"{i:2}. {q}")

## 20. Security Testing

Rangkaian input berbahaya harus **100%** ditolak oleh guardrail. Ini murni deterministik (tidak butuh LLM) karena menguji `validate_sql`/`execute_sql` langsung.

In [ ]:
SECURITY_TESTS = [
    ("drop_table",      "DROP TABLE orders"),
    ("delete_rows",     "DELETE FROM customers"),
    ("update_rows",     "UPDATE products SET unit_price = 0"),
    ("stacked_query",   "SELECT * FROM orders; DROP TABLE orders"),
    ("unknown_table",   "SELECT * FROM secret_table"),
    ("copy_out",        "COPY orders TO '/tmp/orders.csv'"),
    ("pragma",          "PRAGMA database_list"),
    ("attach_db",       "ATTACH 'evil.db' AS evil"),
    ("create_table",    "CREATE TABLE hack(x INT)"),
    ("insert_rows",     "INSERT INTO customers VALUES (1)"),
]

rows = []
for name, sql in SECURITY_TESTS:
    r = execute_sql(sql)
    blocked = (r.status == "error" and r.error_type == "UnsafeSQL")
    rows.append({"test_name": name, "input_sql": sql[:45],
                 "expected": "blocked", "actual": ("blocked" if blocked else r.status),
                 "passed": blocked})
sec_df = pd.DataFrame(rows)
print(sec_df.to_string(index=False))
_passed = int(sec_df.passed.sum())
print(f"\nSecurity: {_passed}/{len(sec_df)} lulus  ->  {_passed/len(sec_df)*100:.0f}%")
assert _passed == len(sec_df), "Ada security test yang gagal!"
print("TARGET 100% TERCAPAI.")

## 21. Uji Self-Correction

Menguji bahwa loop koreksi memperbaiki SQL yang error. Butuh LLM; jika nonaktif, di-skip. Kita berikan pertanyaan yang cenderung memancing kesalahan nama kolom, lalu memeriksa apakah agent berhasil pada percobaan ke-2/3.

In [ ]:
if LLM_ENABLED:
    tricky = "Tampilkan total pendapatan bersih per kuartal 2025 memakai istilah 'net_revenue'."
    r = run_sales_agent(tricky, show_trace=True)
    print("\nHasil self-correction: status=", r.status, "retries=", r.retries)
else:
    print("LLM nonaktif — uji self-correction dilewati.")

## 22. Evaluation Dataset (50 Pertanyaan)

Lima puluh pertanyaan dengan **ground-truth SQL** yang sudah diverifikasi berjalan di database. Komposisi: agregasi, time-series, perbandingan, ranking, multi-table join, dan edge case. Ground truth memakai view `v_completed_sales` sesuai aturan bisnis.

In [ ]:
# Setiap entri: (id, question, ground_truth_sql, category, difficulty)
EVAL = [
    # --- aggregation (10) ---
    (1,"Total revenue 2025","SELECT SUM(revenue) AS revenue FROM v_completed_sales WHERE year=2025","aggregation","easy"),
    (2,"Total profit 2025","SELECT SUM(profit) AS profit FROM v_completed_sales WHERE year=2025","aggregation","easy"),
    (3,"Jumlah order unik 2025","SELECT COUNT(DISTINCT order_id) AS orders FROM v_completed_sales WHERE year=2025","aggregation","easy"),
    (4,"Total quantity terjual 2024","SELECT SUM(quantity) AS qty FROM v_completed_sales WHERE year=2024","aggregation","easy"),
    (5,"Rata-rata revenue per item 2025","SELECT AVG(revenue) AS avg_rev FROM v_completed_sales WHERE year=2025","aggregation","medium"),
    (6,"Total diskon 2025","SELECT SUM(discount) AS discount FROM v_completed_sales WHERE year=2025","aggregation","easy"),
    (7,"AOV keseluruhan 2025","SELECT SUM(revenue)/COUNT(DISTINCT order_id) AS aov FROM v_completed_sales WHERE year=2025","aggregation","medium"),
    (8,"Total revenue seluruh periode","SELECT SUM(revenue) AS revenue FROM v_completed_sales","aggregation","easy"),
    (9,"Jumlah pelanggan aktif 2025","SELECT COUNT(DISTINCT customer_id) AS customers FROM v_completed_sales WHERE year=2025","aggregation","medium"),
    (10,"Total profit seluruh periode","SELECT SUM(profit) AS profit FROM v_completed_sales","aggregation","easy"),
    # --- time-series (10) ---
    (11,"Revenue per bulan 2025","SELECT month, SUM(revenue) AS revenue FROM v_completed_sales WHERE year=2025 GROUP BY month ORDER BY month","time-series","medium"),
    (12,"Revenue per kuartal 2025","SELECT quarter, SUM(revenue) AS revenue FROM v_completed_sales WHERE year=2025 GROUP BY quarter ORDER BY quarter","time-series","medium"),
    (13,"Revenue per tahun","SELECT year, SUM(revenue) AS revenue FROM v_completed_sales GROUP BY year ORDER BY year","time-series","easy"),
    (14,"Profit per bulan 2024","SELECT month, SUM(profit) AS profit FROM v_completed_sales WHERE year=2024 GROUP BY month ORDER BY month","time-series","medium"),
    (15,"Order per bulan 2025","SELECT month, COUNT(DISTINCT order_id) AS orders FROM v_completed_sales WHERE year=2025 GROUP BY month ORDER BY month","time-series","medium"),
    (16,"Quantity per kuartal 2025","SELECT quarter, SUM(quantity) AS qty FROM v_completed_sales WHERE year=2025 GROUP BY quarter ORDER BY quarter","time-series","medium"),
    (17,"Revenue per bulan 2024","SELECT month, SUM(revenue) AS revenue FROM v_completed_sales WHERE year=2024 GROUP BY month ORDER BY month","time-series","medium"),
    (18,"AOV per kuartal 2025","SELECT quarter, SUM(revenue)/COUNT(DISTINCT order_id) AS aov FROM v_completed_sales WHERE year=2025 GROUP BY quarter ORDER BY quarter","time-series","hard"),
    (19,"Revenue kumulatif per bulan 2025","SELECT month, SUM(SUM(revenue)) OVER (ORDER BY month) AS cum_rev FROM v_completed_sales WHERE year=2025 GROUP BY month ORDER BY month","time-series","hard"),
    (20,"Profit per tahun","SELECT year, SUM(profit) AS profit FROM v_completed_sales GROUP BY year ORDER BY year","time-series","easy"),
    # --- comparison (10) ---
    (21,"Revenue Q1 vs Q2 2025","SELECT quarter, SUM(revenue) AS revenue FROM v_completed_sales WHERE year=2025 AND quarter IN (1,2) GROUP BY quarter ORDER BY quarter","comparison","medium"),
    (22,"Revenue 2024 vs 2025","SELECT year, SUM(revenue) AS revenue FROM v_completed_sales GROUP BY year ORDER BY year","comparison","easy"),
    (23,"Revenue per sales channel 2025","SELECT sales_channel, SUM(revenue) AS revenue FROM v_completed_sales WHERE year=2025 GROUP BY sales_channel ORDER BY revenue DESC","comparison","medium"),
    (24,"AOV per sales channel 2025","SELECT sales_channel, SUM(revenue)/COUNT(DISTINCT order_id) AS aov FROM v_completed_sales WHERE year=2025 GROUP BY sales_channel ORDER BY aov DESC","comparison","hard"),
    (25,"Revenue per payment method 2025","SELECT payment_method, SUM(revenue) AS revenue FROM v_completed_sales WHERE year=2025 GROUP BY payment_method ORDER BY revenue DESC","comparison","medium"),
    (26,"Profit Q3 vs Q4 2025","SELECT quarter, SUM(profit) AS profit FROM v_completed_sales WHERE year=2025 AND quarter IN (3,4) GROUP BY quarter ORDER BY quarter","comparison","medium"),
    (27,"Website vs Marketplace revenue 2025","SELECT sales_channel, SUM(revenue) AS revenue FROM v_completed_sales WHERE year=2025 AND sales_channel IN ('Website','Marketplace') GROUP BY sales_channel","comparison","medium"),
    (28,"Revenue per kategori 2025","SELECT c.category_name, SUM(v.revenue) AS revenue FROM v_completed_sales v JOIN products p ON v.product_id=p.product_id JOIN categories c ON p.category_id=c.category_id WHERE v.year=2025 GROUP BY c.category_name ORDER BY revenue DESC","comparison","hard"),
    (29,"Revenue per region 2025","SELECT cu.region, SUM(v.revenue) AS revenue FROM v_completed_sales v JOIN customers cu ON v.customer_id=cu.customer_id WHERE v.year=2025 GROUP BY cu.region ORDER BY revenue DESC","comparison","hard"),
    (30,"Quantity 2024 vs 2025","SELECT year, SUM(quantity) AS qty FROM v_completed_sales GROUP BY year ORDER BY year","comparison","easy"),
    # --- ranking (8) ---
    (31,"5 produk revenue terbesar 2025","SELECT p.product_name, SUM(v.revenue) AS revenue FROM v_completed_sales v JOIN products p ON v.product_id=p.product_id WHERE v.year=2025 GROUP BY p.product_name ORDER BY revenue DESC LIMIT 5","ranking","hard"),
    (32,"10 pelanggan pembelian terbesar 2025","SELECT cu.customer_name, SUM(v.revenue) AS revenue FROM v_completed_sales v JOIN customers cu ON v.customer_id=cu.customer_id WHERE v.year=2025 GROUP BY cu.customer_name ORDER BY revenue DESC LIMIT 10","ranking","hard"),
    (33,"Kategori profit tertinggi 2025","SELECT c.category_name, SUM(v.profit) AS profit FROM v_completed_sales v JOIN products p ON v.product_id=p.product_id JOIN categories c ON p.category_id=c.category_id WHERE v.year=2025 GROUP BY c.category_name ORDER BY profit DESC LIMIT 1","ranking","hard"),
    (34,"Region revenue tertinggi 2025","SELECT cu.region, SUM(v.revenue) AS revenue FROM v_completed_sales v JOIN customers cu ON v.customer_id=cu.customer_id WHERE v.year=2025 GROUP BY cu.region ORDER BY revenue DESC LIMIT 1","ranking","hard"),
    (35,"5 kategori revenue terbesar seluruh periode","SELECT c.category_name, SUM(v.revenue) AS revenue FROM v_completed_sales v JOIN products p ON v.product_id=p.product_id JOIN categories c ON p.category_id=c.category_id GROUP BY c.category_name ORDER BY revenue DESC LIMIT 5","ranking","hard"),
    (36,"10 produk quantity terbesar 2025","SELECT p.product_name, SUM(v.quantity) AS qty FROM v_completed_sales v JOIN products p ON v.product_id=p.product_id WHERE v.year=2025 GROUP BY p.product_name ORDER BY qty DESC LIMIT 10","ranking","hard"),
    (37,"3 channel revenue terbesar 2025","SELECT sales_channel, SUM(revenue) AS revenue FROM v_completed_sales WHERE year=2025 GROUP BY sales_channel ORDER BY revenue DESC LIMIT 3","ranking","medium"),
    (38,"5 kota revenue terbesar 2025","SELECT cu.city, SUM(v.revenue) AS revenue FROM v_completed_sales v JOIN customers cu ON v.customer_id=cu.customer_id WHERE v.year=2025 GROUP BY cu.city ORDER BY revenue DESC LIMIT 5","ranking","hard"),
    # --- multi-table join (7) ---
    (39,"Profit per kategori 2025","SELECT c.category_name, SUM(v.profit) AS profit FROM v_completed_sales v JOIN products p ON v.product_id=p.product_id JOIN categories c ON p.category_id=c.category_id WHERE v.year=2025 GROUP BY c.category_name ORDER BY profit DESC","join","hard"),
    (40,"Revenue per segment pelanggan 2025","SELECT cu.customer_segment, SUM(v.revenue) AS revenue FROM v_completed_sales v JOIN customers cu ON v.customer_id=cu.customer_id WHERE v.year=2025 GROUP BY cu.customer_segment ORDER BY revenue DESC","join","hard"),
    (41,"Revenue kategori per region 2025","SELECT cu.region, c.category_name, SUM(v.revenue) AS revenue FROM v_completed_sales v JOIN customers cu ON v.customer_id=cu.customer_id JOIN products p ON v.product_id=p.product_id JOIN categories c ON p.category_id=c.category_id WHERE v.year=2025 GROUP BY cu.region, c.category_name ORDER BY revenue DESC LIMIT 20","join","hard"),
    (42,"AOV per region 2025","SELECT cu.region, SUM(v.revenue)/COUNT(DISTINCT v.order_id) AS aov FROM v_completed_sales v JOIN customers cu ON v.customer_id=cu.customer_id WHERE v.year=2025 GROUP BY cu.region ORDER BY aov DESC","join","hard"),
    (43,"Profit per region 2025","SELECT cu.region, SUM(v.profit) AS profit FROM v_completed_sales v JOIN customers cu ON v.customer_id=cu.customer_id WHERE v.year=2025 GROUP BY cu.region ORDER BY profit DESC","join","hard"),
    (44,"Revenue kategori per channel 2025","SELECT v.sales_channel, c.category_name, SUM(v.revenue) AS revenue FROM v_completed_sales v JOIN products p ON v.product_id=p.product_id JOIN categories c ON p.category_id=c.category_id WHERE v.year=2025 GROUP BY v.sales_channel, c.category_name ORDER BY revenue DESC LIMIT 20","join","hard"),
    (45,"Jumlah produk per kategori","SELECT c.category_name, COUNT(*) AS n_products FROM products p JOIN categories c ON p.category_id=c.category_id GROUP BY c.category_name ORDER BY n_products DESC","join","medium"),
    # --- edge case (5) ---
    (46,"Revenue kategori Automotive 2025 (mungkin kecil/kosong)","SELECT c.category_name, SUM(v.revenue) AS revenue FROM v_completed_sales v JOIN products p ON v.product_id=p.product_id JOIN categories c ON p.category_id=c.category_id WHERE v.year=2025 AND c.category_name='Automotive' GROUP BY c.category_name","edge","medium"),
    (47,"Revenue tahun 2030 (tidak ada data)","SELECT SUM(revenue) AS revenue FROM v_completed_sales WHERE year=2030","edge","medium"),
    (48,"Order dengan revenue negatif (seharusnya tidak ada)","SELECT COUNT(*) AS n FROM v_completed_sales WHERE revenue < 0","edge","medium"),
    (49,"Pelanggan tanpa transaksi completed","SELECT COUNT(*) AS n FROM customers cu WHERE cu.customer_id NOT IN (SELECT DISTINCT customer_id FROM v_completed_sales)","edge","hard"),
    (50,"Revenue bulan 13 (invalid, harus kosong)","SELECT SUM(revenue) AS revenue FROM v_completed_sales WHERE month=13","edge","medium"),
]
eval_df = pd.DataFrame(EVAL, columns=["id","question","ground_truth_sql","category","difficulty"])
print("Eval dataset:", eval_df.shape)
print(eval_df.category.value_counts().to_string())

### 22.1 Verifikasi ground truth berjalan (deterministik)

Setiap ground-truth SQL harus lolos guardrail dan tereksekusi. Ini menjamin dataset evaluasi valid sebelum dipakai menilai agent.

In [ ]:
_gt_ok = 0
_gt_fail = []
for _, row in eval_df.iterrows():
    r = execute_sql(row.ground_truth_sql)
    if r.status == "success":
        _gt_ok += 1
    else:
        _gt_fail.append((row.id, r.error_type, r.error_message))
print(f"Ground truth valid: {_gt_ok}/{len(eval_df)}")
if _gt_fail:
    print("GAGAL:", _gt_fail)
assert _gt_ok == len(eval_df), "Ada ground truth yang tidak jalan!"
print("Semua ground truth tereksekusi.")

## 23. Evaluation Runner & 24. Metrik

**Execution accuracy**: hasil SQL agent dibandingkan dengan hasil ground truth berdasarkan **nilai**, bukan teks SQL. Untuk adil, hasil di-*canonicalize*: ambil nilai numerik, dibulatkan 2 desimal, diurutkan, dibandingkan sebagai multiset. Ini menghilangkan false-negative akibat beda urutan baris/kolom atau presisi float.

In [ ]:
def canonicalize(df: pd.DataFrame):
    if df is None:
        return None
    vals = []
    for col in df.columns:
        s = df[col]
        if pd.api.types.is_numeric_dtype(s):
            vals.extend(sorted(round(float(x), 2) for x in s.dropna().tolist()))
        else:
            vals.extend(sorted(str(x) for x in s.dropna().tolist()))
    return tuple(sorted(map(str, vals)))

def results_match(df_a, df_b):
    return canonicalize(df_a) == canonicalize(df_b)

def run_evaluation(sample: Optional[int] = None):
    if not LLM_ENABLED:
        print("LLM nonaktif — evaluasi dilewati.")
        return None
    data = eval_df if sample is None else eval_df.head(sample)
    rows = []
    for _, row in data.iterrows():
        gt = execute_sql(row.ground_truth_sql)
        res = run_sales_agent(row.question, show_trace=False)
        agent_df = res.dataframe if res.status == "success" else None
        correct = (res.status == "success" and gt.status == "success"
                   and results_match(agent_df, gt.dataframe))
        rows.append({
            "id": row.id, "category": row.category, "difficulty": row.difficulty,
            "agent_status": res.status, "correct": bool(correct),
            "retries": res.retries, "latency": round(res.latency_total,2),
            "in_tok": res.input_tokens, "out_tok": res.output_tokens,
            "cost": round(res.est_cost,6),
            "faithfulness": (res.faithfulness or {}).get("score"),
        })
    return pd.DataFrame(rows)

print("Evaluation runner siap. Panggil run_evaluation() atau run_evaluation(sample=10).")

In [ ]:
if LLM_ENABLED:
    eval_result = run_evaluation()   # jalankan 50 pertanyaan
    n = len(eval_result)
    exec_acc = eval_result.correct.mean()
    valid_sql_rate = (eval_result.agent_status == "success").mean()
    avg_retry = eval_result.retries.mean()
    med_latency = eval_result.latency.median()
    total_cost = eval_result.cost.sum()
    faith = eval_result.faithfulness.dropna().mean() if eval_result.faithfulness.notna().any() else None

    print("="*60)
    print("LAPORAN EVALUASI")
    print("="*60)
    print(f"Execution Accuracy   : {exec_acc*100:.1f}%   (target >= 80%)")
    print(f"Valid SQL Rate       : {valid_sql_rate*100:.1f}%   (target >= 90%)")
    print(f"Average Retry        : {avg_retry:.2f}      (target <= 0.5)")
    print(f"Median Latency       : {med_latency:.2f}s    (target <= 10s)")
    print(f"Faithfulness (avg)   : {('%.2f'%faith) if faith is not None else 'n/a'}")
    print(f"Total Cost (50 q)    : ${total_cost:.4f}")
    print("\nAkurasi per kategori:")
    print((eval_result.groupby('category').correct.mean()*100).round(1).to_string())
    print("\nAkurasi per difficulty:")
    print((eval_result.groupby('difficulty').correct.mean()*100).round(1).to_string())
else:
    print("LLM nonaktif — laporan evaluasi tidak dibuat. Aktifkan OPENAI_API_KEY lalu jalankan ulang bagian ini.")

## 25. Error Analysis

Bila ada pertanyaan yang salah, kelompokkan penyebabnya untuk perbaikan terarah.

In [ ]:
if LLM_ENABLED and 'eval_result' in dir():
    wrong = eval_result[~eval_result.correct]
    if len(wrong):
        print("Pertanyaan yang belum benar:")
        print(wrong[["id","category","difficulty","agent_status","retries"]].to_string(index=False))
        print("\nKategori error kandidat: wrong_join, wrong_filter, wrong_aggregation,")
        print("wrong_business_definition, empty_result, sql_syntax, timeout.")
    else:
        print("Tidak ada error — seluruh pertanyaan benar.")
else:
    print("Jalankan evaluasi (LLM aktif) untuk analisis error.")

# Trace keseluruhan dapat dikonversi ke DataFrame
if agent_traces:
    print("\nContoh trace terakhir:")
    print(pd.DataFrame(agent_traces).tail(3).to_string(index=False))

## 26. Keterbatasan

- **Bukan ReAct agent**, melainkan *bounded pipeline* deterministik (understand → generate → validate → execute → correct). Untuk domain SQL, keandalan lebih penting daripada kebebasan perencanaan. Ini pilihan desain sadar, bukan kekurangan.
- **Data sintetis** — pola dibuat manual; tidak mewakili dinamika bisnis nyata.
- **Faithfulness check berbasis pencocokan angka** — pendekatan heuristik; tidak menangkap kesalahan interpretasi kualitatif.
- **Execution accuracy** bergantung pada kualitas ground truth; pertanyaan ambigu bisa memiliki lebih dari satu jawaban benar.
- **Biaya/latency** bergantung pada provider LLM yang dipakai.

## 27. Roadmap

1. **Fase 1 (notebook ini)** — prototipe end-to-end.
2. **Fase 2** — modularisasi ke paket Python (`agent/`, `tools/`, `guardrails/`, `database/`, `visualization/`, `evaluation/`).
3. **Fase 3** — aplikasi web (FastAPI + Streamlit/Next.js), PostgreSQL, Redis.
4. **Fase 4** — produksi: auth, RBAC, row-level security, audit log, prompt versioning, semantic layer, model fallback, monitoring, cost budget, container.

## 28. Kesimpulan

SalesInsight Agent menunjukkan cara menghubungkan LLM ke database secara **aman dan terukur**: aturan bisnis ditegakkan di lapisan data (view), SQL divalidasi lewat AST sebelum dieksekusi, koneksi read-only plus timeout membatasi dampak, self-correction menaikkan keberhasilan, dan evaluasi berbasis *execution accuracy* memberi ukuran kualitas yang jujur. Pola ini dapat ditingkatkan menjadi aplikasi produksi tanpa mengubah prinsip intinya: **pipa deterministik dengan LLM sebagai komponen, bukan pengambil keputusan yang tak terkendali.**